# Assignment 2: Steady-state

**Author:** Lino Kaiser.

This notebook validates the thermal circuit from [01_model.ipynb](01_model.ipynb). It follows the DAE and steady-state procedure in the supplied [cubic-building tutorial](https://github.com/cghiaus/dm4bem/blob/2eb8803cf038bd73c1618f2fd89771b9ba285ab4/03CubicBuilding.ipynb).


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

import dm4bem
from building import definition

building = definition["building"]
zones = definition["zones"]
openings = definition["openings"]
materials = definition["materials"]
assemblies = definition["assemblies"]
boundary = definition["boundary"]

zone_names = ["project_room", "office"]
air_nodes = [0, 7]
rho_air = 1.2       # kg/m³, assumed
cp_air = 1005       # J/(kg K), assumed
r_inside = boundary["inside_combined_film_R_m2K_W"]
r_outside = boundary["outside_combined_film_R_m2K_W"]
h_convection = 3.0
h_radiation = 1 / r_inside - h_convection

from thermal_model import (
    areas, door_area, partition_area, node_names, capacities, branches,
    ventilation_branches, heating_branches, partition_branches, internal_door_branch,
)


## 1. DAE and state-space representation

We use the DAE and sign convention defined in [notebook 1, section 9](01_model.ipynb). The supplied `tc2ss` function converts this model to state-space form.


In [2]:
from thermal_model import A, base_conductance, b, G, C, f, y, n_nodes, n_branches

As, Bs, Cs, Ds = dm4bem.tc2ss(A, G, C, b, f, y)
state_nodes = np.flatnonzero(capacities)
input_names = (
    [f"b_T[{j}]: {branches[j][1]}" for j in np.flatnonzero(b)]
    + [f"f_Q[{j}]: {node_names[j]}" for j in np.flatnonzero(f)]
)

print(f"DAE: A {A.shape}, G {G.shape}, C {C.shape}")
print(f"State space: As {As.shape}, Bs {Bs.shape}, Cs {Cs.shape}, Ds {Ds.shape}")
print("State temperature nodes:", state_nodes)
print("b =", b)
print("f =", f)
print("y =", y)
display(pd.DataFrame({"Input": input_names}))

DAE: A (34, 15), G (34, 34), C (15, 15)
State space: As (9, 9), Bs (9, 15), Cs (2, 9), Ds (2, 15)
State temperature nodes: [ 0  1  3  5  7  8 10 12 14]
b = [1 0 0 0 1 0 0 0 1 0 0 0 1 1 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 1 1 0 0 0]
f = [1 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
y = [1 0 0 0 0 0 0 1 0 0 0 0 0 0 0]


,Input
0,b_T[0]: outdoor
1,b_T[4]: outdoor
2,b_T[8]: ground
3,b_T[12]: outdoor
4,b_T[13]: outdoor
5,b_T[14]: outdoor
6,b_T[15]: setpoint
7,b_T[16]: outdoor
8,b_T[20]: outdoor
9,b_T[24]: ground


## 2. Independent steady-state heat balance

At steady state, no thermal energy accumulates, so the total heat entering each node equals the total heat leaving it.

Setting $\dot{\theta}=0$ in the DAE gives:

$$0=-A^TGA\theta_\mathrm{ss}+A^TG b_T+f_Q.$$

Rearranging gives the equation used to solve the full circuit in section 3:

$$A^TGA\theta_\mathrm{ss}=A^TG b_T+f_Q.$$

With temperatures constant, each envelope path can then be reduced to its complete series resistance. This gives a separate two-room heat balance, calculated directly from layer thicknesses, material properties and opening U-values.

Let H_o be each room's conductance to outside, H_g its conductance to ground, H_12 the combined partition/door conductance, and K_p the controller gain. For room 1:

$$(H_{o,1}+H_{g,1}+H_{12}+K_p)T_1-H_{12}T_2
=H_{o,1}T_o+H_{g,1}T_g+Q_1+K_pT_\mathrm{set}.$$

Room 2 has the corresponding second equation. These two equations provide a check independent of the numbered circuit and its state-space conversion.

In [3]:
# Direct layer sums, independent of the circuit's split resistances.
u_values = {}
for name, layers in assemblies.items():
    r_layers = sum(d / materials[m]["lambda_W_mK"] for m, d in layers)
    if name in ["partition", "internal_door"]:
        r_films = 2 * r_inside
    elif name == "floor":
        r_films = r_inside
    else:
        r_films = r_inside + r_outside
    u_values[name] = 1 / (r_layers + r_films)

H_outside = np.array([
    u_values["external_wall"] * areas[name]["external_wall"]
    + u_values["roof"] * areas[name]["roof"]
    + boundary["window_U_W_m2K"] * areas[name]["windows"]
    + boundary["external_door_U_W_m2K"] * areas[name]["external_door"]
    for name in zone_names
])
H_ground = np.array([u_values["floor"] * areas[name]["floor"] for name in zone_names])
H_ventilation = np.array([
    rho_air * cp_air * areas[name]["volume"] / 3600 for name in zone_names
])
H_between = (u_values["partition"] * partition_area
             + u_values["internal_door"] * door_area)

display(pd.DataFrame({
    "Envelope to outside [W/K]": H_outside,
    "Ground [W/K]": H_ground,
    "Ventilation at 1 ACH [W/K]": H_ventilation,
}, index=zone_names).round(3))
print(f"Between rooms: {H_between:.3f} W/K")

,Envelope to outside [W/K],Ground [W/K],Ventilation at 1 ACH [W/K]
project_room,32.273,11.148,36.18
office,16.185,5.574,18.09


Between rooms: 35.366 W/K


## 3. Validation cases

We compare three results for each case:

1. Solve the complete steady-state DAE heat balance.
2. Convert with the supplied tc2ss function and solve the state-space equilibrium.
3. Solve the independent two-room balance above.

We also check every node's energy balance, the global energy balance, the shared-wall/door flow, and the stability of the state matrix.

In [4]:
def solve_case(T_out, T_ground, gains=(0.0, 0.0), ach=1.0, kp=0.0, T_set=21.0):
    conductance = base_conductance.copy()
    conductance[ventilation_branches] *= ach
    conductance[heating_branches] = kp
    G_case = np.diag(conductance)

    sources = {"outdoor": T_out, "ground": T_ground, "setpoint": T_set}
    b_T = np.array([sources[start] if isinstance(start, str) else 0.0
                    for _, start, _, _ in branches])
    f_Q = np.zeros(n_nodes)
    f_Q[air_nodes] = gains
    u = np.concatenate((b_T[b == 1], f_Q[f == 1]))

    theta = np.linalg.solve(A.T @ G_case @ A, A.T @ G_case @ b_T + f_Q)
    As_case, Bs_case, Cs_case, Ds_case = dm4bem.tc2ss(A, G_case, C, b, f, y)
    x_ss = -np.linalg.solve(As_case, Bs_case @ u)
    y_ss = Cs_case @ x_ss + Ds_case @ u

    H_out = H_outside + ach * H_ventilation
    H = np.diag(H_out + H_ground + H_between + kp)
    H[0, 1] = H[1, 0] = -H_between
    rhs = H_out * T_out + H_ground * T_ground + np.array(gains) + kp * T_set
    reference = np.linalg.solve(H, rhs)

    q = G_case @ (-A @ theta + b_T)
    node_balance = A.T @ q + f_Q
    global_balance = q[b == 1].sum() + f_Q.sum()
    shared_flow = q[partition_branches[0]] + q[internal_door_branch]

    np.testing.assert_allclose(theta[air_nodes], y_ss, atol=1e-8, rtol=0)
    np.testing.assert_allclose(theta[air_nodes], reference, atol=1e-8, rtol=0)
    np.testing.assert_allclose(theta[state_nodes], x_ss, atol=1e-8, rtol=0)
    np.testing.assert_allclose(node_balance, 0, atol=1e-7, rtol=0)
    np.testing.assert_allclose(global_balance, 0, atol=1e-7, rtol=0)
    np.testing.assert_allclose(shared_flow, H_between * (reference[0] - reference[1]),
                               atol=1e-7, rtol=0)
    assert np.linalg.eigvals(As_case).real.max() < 0
    if kp > 0:
        assert np.all(q[heating_branches] >= -1e-7), "This case requires cooling."

    return {
        "T1 DAE [°C]": theta[0], "T2 DAE [°C]": theta[7],
        "T1 state space [°C]": y_ss[0], "T2 state space [°C]": y_ss[1],
        "T1 reference [°C]": reference[0], "T2 reference [°C]": reference[1],
        "DAE/SS error [K]": np.max(np.abs(theta[air_nodes] - y_ss)),
        "Reference error [K]": np.max(np.abs(theta[air_nodes] - reference)),
        "Node balance error [W]": np.max(np.abs(node_balance)),
        "Project to office [W]": shared_flow,
    }


occupied_gains = tuple(
    zones[name]["people"] * boundary["person_sensible_W"]
    + zones[name]["laptops"] * boundary["laptop_W"]
    + areas[name]["floor"] * boundary["lighting_W_m2"]
    for name in zone_names
)
cases = {
    "Uniform 20 °C": dict(T_out=20, T_ground=20, T_set=20),
    "Winter, free-running": dict(T_out=0, T_ground=10, ach=0.3),
    "Winter, occupied": dict(T_out=0, T_ground=10, gains=occupied_gains),
    "Winter, heating": dict(T_out=0, T_ground=10, gains=occupied_gains, kp=500),
    "Project-only heat": dict(T_out=0, T_ground=0, gains=(1000, 0)),
}
results = pd.DataFrame({name: solve_case(**case) for name, case in cases.items()}).T
np.testing.assert_allclose(results.loc["Uniform 20 °C"].iloc[:6], 20, atol=1e-8)
assert results.loc["Project-only heat", "T1 DAE [°C]"] > results.loc[
    "Project-only heat", "T2 DAE [°C]"
] > 0
display(results.iloc[:, :6].round(4))
display(results.iloc[:, 6:].round(10))
print("All five steady-state cases passed.")

,T1 DAE [°C],T2 DAE [°C],T1 state space [°C],T2 state space [°C],T1 reference [°C],T2 reference [°C]
Uniform 20 °C,20.0000,20.0000,20.0000,20.0000,20.0000,20.0000
"Winter, free-running",2.0532,2.0520,2.0532,2.0520,2.0532,2.0520
"Winter, occupied",9.9408,9.9357,9.9408,9.9357,9.9408,9.9357
"Winter, heating",19.5195,20.1422,19.5195,20.1422,19.5195,20.1422
Project-only heat,10.1690,4.7815,10.1690,4.7815,10.1690,4.7815


,DAE/SS error [K],Reference error [K],Node balance error [W],Project to office [W]
Uniform 20 °C,0.0,0.0,0.0,0.000000
"Winter, free-running",0.0,0.0,0.0,0.043390
"Winter, occupied",0.0,0.0,0.0,0.181440
"Winter, heating",0.0,0.0,0.0,-22.022645
Project-only heat,0.0,0.0,0.0,190.535417


All five steady-state cases passed.


## 4. Heating demand at 21 °C

For an ideal 21 °C in both rooms, the interzone temperature difference is zero. Required heating is therefore each room's outdoor and ground heat loss minus its internal gains.

The following is an occupied winter design point: outdoor 0 °C, ground 10 °C, 1 ACH, no sun, and the defined people/equipment/lighting gains. It is a steady-state for this model.

In [5]:
setpoint = boundary["occupied_setpoint_C"]
outdoor_loss = (H_outside + H_ventilation) * (setpoint - 0)
ground_loss = H_ground * (setpoint - boundary["ground_temperature_C"])
heating = outdoor_loss + ground_loss - np.array(occupied_gains)
display(pd.DataFrame({
    "Outdoor loss [W]": outdoor_loss,
    "Ground loss [W]": ground_loss,
    "Internal gains [W]": occupied_gains,
    "Required heating [W]": heating,
}, index=zone_names).round(1))

,Outdoor loss [W],Ground loss [W],Internal gains [W],Required heating [W]
project_room,1437.5,122.6,680.0,880.1
office,719.8,61.3,340.0,441.1


## 5. Conclusion

- The uniform-temperature case returns 20 °C in both rooms, with zero net heat exchange. Which is what we want/expect, ergo it's good.
- Without heating or internal gains, the prescribed 10 °C ground supplies some heat while the 0 °C exterior removes it.
- Occupied-period gains raise the free-running room temperatures. Heating raises them further.
- Heating only the project room also warms the office through the shared partition and door.
- The DAE, state-space and independent two-room calculations all agree, and the nodal/global heat balances work out.

**Sources:** Christian Ghiaus, [03CubicBuilding.ipynb](https://github.com/cghiaus/dm4bem/blob/2eb8803cf038bd73c1618f2fd89771b9ba285ab4/03CubicBuilding.ipynb).
